# MLflow → RO-Crate

MLflow already tracked your experiment: parameters, metrics, the dataset that went in,
the model and files that came out, who ran it. This tool turns that record into a FAIRSCAPE RO-Crate.

1. **Train** a model with MLflow
2. **Convert** the tracking store into an RO-Crate.
3. **View** the automatically generated datasheet and the provenance graph
3. **Larger Graph** View a more detailed run and the provenance associated with it

```bash
pip install mlflow scikit-learn pandas fairscape-cli
pip install -e ../..            # fairscape-conversion
```

## Setup

In [ ]:
import json, os, shutil, warnings, html
from pathlib import Path
from IPython.display import HTML

os.environ["MLFLOW_LOGGING_LEVEL"] = "ERROR"             
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"
warnings.filterwarnings("ignore", category=UserWarning)

import mlflow, mlflow.sklearn
import pandas as pd
from fairscape_conversion.plugins import mlflow as to_rocrate

TRACKING = "sqlite:///mlflow.db"     # any MLflow tracking URI works here
CRATE = Path("crate")                # where the RO-Crate lands
WORK = Path("work"); WORK.mkdir(exist_ok=True)

def show(path, height=700):
    """Render a self-contained HTML file inside the notebook."""
    doc = html.escape(Path(path).read_text())
    return HTML(f'<iframe srcdoc="{doc}" width="100%" height="{height}" '
                'style="border:1px solid #ddd;border-radius:6px"></iframe>')

## 1. Train

Plain MLflow. Everything logged here shows up in the crate:

| you log | the crate gets |
|---|---|
| `log_input` | a Dataset with a Schema from the column spec |
| `log_params`, `log_metrics` | parameters and metrics on the Computation |
| `log_artifact` | a Dataset, `generatedBy` the run |
| `log_model` | an `MLModel`, with `trainedOn` |
| a nested run | a Computation that `isPartOf` its parent |

In [2]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

Path("mlflow.db").unlink(missing_ok=True); shutil.rmtree("mlruns", ignore_errors=True)
mlflow.set_tracking_uri(TRACKING)
mlflow.set_experiment("iris-classifier")

features, target = load_iris(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.25, random_state=0, stratify=target)

with mlflow.start_run(run_name="rf-baseline"):
    mlflow.log_input(mlflow.data.from_pandas(features.assign(species=target),
                                             name="iris", targets="species"))
    mlflow.log_params({"n_estimators": 200, "max_depth": 4, "random_state": 0})

    model = RandomForestClassifier(n_estimators=200, max_depth=4, random_state=0)
    model.fit(X_train, y_train)
    predicted = model.predict(X_test)
    mlflow.log_metrics({"accuracy": accuracy_score(y_test, predicted),
                        "f1_macro": f1_score(y_test, predicted, average="macro")})

    pd.DataFrame(confusion_matrix(y_test, predicted)).to_csv(WORK / "confusion_matrix.csv", index=False)
    mlflow.log_artifact(WORK / "confusion_matrix.csv")
    mlflow.sklearn.log_model(model, name="model", input_example=X_train.head(3),
                             serialization_format="pickle")     # ~10x smaller than skops

    with mlflow.start_run(run_name="feature-importance", nested=True):
        importance = pd.DataFrame({"feature": features.columns, "importance": model.feature_importances_})
        importance.sort_values("importance", ascending=False).to_csv(WORK / "feature_importance.csv", index=False)
        mlflow.log_artifact(WORK / "feature_importance.csv")
        mlflow.log_metric("top_importance", importance.importance.max())

print(f"accuracy {accuracy_score(y_test, predicted):.3f}")

accuracy 0.947


## 2. Convert

Point the converter at the tracking store. It reads the runs, copies the artifacts and
the model into `crate/`, and returns the RO-Crate. The same call works for an
`mlruns/` directory or a remote `http://` tracking server.

In [3]:
shutil.rmtree(CRATE, ignore_errors=True)

crate = to_rocrate.convert(
    "import", TRACKING,
    experiment="iris-classifier",
    crate_dir=CRATE,
    naan="59853",
    name="Iris classifier — MLflow experiment",
    description="Random forest trained on the iris dataset, tracked in MLflow "
                "and converted to an EVI RO-Crate.",
    author="Example Researcher",
    keywords="mlflow, iris, random forest, provenance",
    license="https://spdx.org/licenses/CC-BY-4.0",
    schemas=True,
)

(CRATE / "ro-crate-metadata.json").write_text(json.dumps(crate, indent=2, default=str))
print(f"{len(crate['@graph'])} nodes -> crate/ro-crate-metadata.json")

13 nodes -> crate/ro-crate-metadata.json


## 3. The datasheet

`fairscape-cli` builds a human-readable datasheet from the crate. Nothing below was typed by hand.

In [4]:
!fairscape-cli build datasheet crate --skip-subcrate-processing
show(CRATE / "ro-crate-datasheet.html")

Checking subcrate links...

Generating Link-ML for crate/ro-crate-metadata.json
✓ LinkML: crate/ro-crate-linkml.yaml

Generating datasheet for crate/ro-crate-metadata.json
Outputting to: crate/ro-crate-datasheet.html


✓ HTML datasheet: crate/ro-crate-datasheet.html


## 4. The provenance graph

Start from the model and walk backwards: the run that produced it, the data and software the run used.

In [5]:
model_id = next(n["@id"] for n in crate["@graph"] if "MLModel" in str(n["@type"]))

!fairscape-cli build evidence-graph crate {model_id}
show(CRATE / "provenance-graph.html", height=500)

Generating evidence graph for ark:59853/dataset-model-4bcee6d from crate/ro-crate-metadata.json...
Evidence graph saved to crate/provenance-graph.json
Generating visualization...
Visualization saved to crate/provenance-graph.html
Added hasEvidenceGraph reference to ark:59853/dataset-model-4bcee6d in RO-Crate metadata


## 5. Is it valid?

`fairscape_models` holds the same pydantic models the FAIRSCAPE services use.

In [6]:
from fairscape_models.rocrate import ROCrateV1_2

ROCrateV1_2.model_validate(json.loads((CRATE / "ro-crate-metadata.json").read_text()))
print("valid RO-Crate")

for path in sorted(p for p in CRATE.rglob("*") if p.is_file()):
    print(f"{path.stat().st_size:>9,}  {path.relative_to(CRATE)}")

valid RO-Crate
    4,875  ai_ready_score.json
      170  feature-importance-9139d1b0/feature_importance.csv
  293,605  provenance-graph.html
    2,583  provenance-graph.json
       27  rf-baseline-9afb7e48/confusion_matrix.csv
    1,310  rf-baseline-9afb7e48/model/MLmodel
      244  rf-baseline-9afb7e48/model/conda.yaml
      171  rf-baseline-9afb7e48/model/input_example.json
  282,272  rf-baseline-9afb7e48/model/model.pkl
      123  rf-baseline-9afb7e48/model/python_env.yaml
       37  rf-baseline-9afb7e48/model/registered_model_meta
      118  rf-baseline-9afb7e48/model/requirements.txt
      382  rf-baseline-9afb7e48/model/serving_input_example.json
   49,117  ro-crate-datasheet.html
      487  ro-crate-linkml.yaml
   14,318  ro-crate-metadata.json


## 6. More steps, more detailed graph

This 2nd example is three scripts, each its own MLflow run:
**prepare** splits the data, **train** fits a small grid of random forest,
**evaluate** scores that model on the held-out split. The evaluate run declares the model as an input,
which becomes `usedMLModel`, and every step reads the files the previous one wrote.
The graph below starts at the predictions and walks all the way back to the raw data.

In [7]:
!python pipeline/run_pipeline.py
show("pipeline/crate/ro-crate-datasheet.html")

prepare-data: 112 train rows, 38 test rows


train: best candidate {'n_estimators': 200, 'max_depth': 4} (oob accuracy 0.946) -> m-28f8e7d21a92491bac291b203fa318ad


evaluate: accuracy 0.947 on 38 test rows


crate: 27 nodes -> crate/ro-crate-metadata.json


Checking subcrate links...

Generating Link-ML for /home/oj/fairscape/fairscape_conversion/examples/mlflow/pipeline/crate/ro-crate-metadata.json
✓ LinkML: /home/oj/fairscape/fairscape_conversion/examples/mlflow/pipeline/crate/ro-crate-linkml.yaml

Generating datasheet for /home/oj/fairscape/fairscape_conversion/examples/mlflow/pipeline/crate/ro-crate-metadata.json
Outputting to: /home/oj/fairscape/fairscape_conversion/examples/mlflow/pipeline/crate/ro-crate-datasheet.html


✓ HTML datasheet: /home/oj/fairscape/fairscape_conversion/examples/mlflow/pipeline/crate/ro-crate-datasheet.html


Generating evidence graph for ark:59853/dataset-predictions-csv-c7a6b13 from /home/oj/fairscape/fairscape_conversion/examples/mlflow/pipeline/crate/ro-crate-metadata.json...
Evidence graph saved to /home/oj/fairscape/fairscape_conversion/examples/mlflow/pipeline/crate/provenance-graph.json
Generating visualization...
Visualization saved to /home/oj/fairscape/fairscape_conversion/examples/mlflow/pipeline/crate/provenance-graph.html
Added hasEvidenceGraph reference to ark:59853/dataset-predictions-csv-c7a6b13 in RO-Crate metadata


In [8]:
show("pipeline/crate/provenance-graph.html")

## Next

- **One run instead of a whole experiment:** `run_id="…"` instead of `experiment=`.
- **Leave files where MLflow put them:** `copy_artifacts=False`.
- **Same trick, other tools:** `examples/` has D4D, C2M2, Workflow Run RO-Crate, Cromwell, Snakemake and Croissant.
- **How the mapping works:** `plugins/mlflow/entities.csv` and `properties.csv`. Every field above is one row.